# DSCI 6883 - Descriptive Analytics & Data Integrity
## Discussion 1.6 Follow-Up: The Parts That Did Not Land

**Fall 2026 | Module 1 | Fernando Rubio Garcia**

In [0]:
get_ipython().kernel._abort_queues = lambda *a, **kw: None

### Where this came from

In the 1.6 discussion I asked you to point at the exact places where the Python Foundations course lost you. You did, and precise questions make precise answers possible: every section below answers one of your posts.

If your part is in here and it still does not click, reply in the discussion with the **section number and the cell** where it stopped making sense.

### How to use this notebook

1. **Run every cell yourself**, top to bottom. Use `Shift + Enter`.
2. **Predict the output before you run a cell.** Two of you wrote some version of "I can follow the examples, but I could not write it myself." Predicting is the step between those two.
3. **Do the `Your Turn` exercises.** Each one has a solution you can expand - try it first.
4. Some cells are **designed to fail**. They are marked `SUPPOSED TO FAIL`. Read the error message, then keep going.
5. If the notebook starts behaving strangely, use **Kernel > Restart Kernel and Run All Cells**. Section 5 explains what that actually does, and why it works.

### Outline

| Section | Topic | From the post by |
|---|---|---|
| 1 | Which variables a function can see (scope) | Tyler |
| 2 | List comprehensions: from reading them to writing them | Tyler |
| 3 | `enumerate`, dictionaries, dictionary comprehensions, and `import numpy as np` | Jacob |
| 4 | `lambda` and `map()` | Matei |
| 5 | The Kernel menu: what each option does and when to use it | Kara |
| 6 | Saving, finding, and importing your own modules (and what happened to Kara's tax calculator) | Matei, Kara |

> **Section 6 creates a few small files** whose names start with `demo_` in the same folder as this notebook. The last cell deletes them.

> **Working in Databricks, like Scott?** Sections 1-4 are plain Python. Sections 5 and 6 are specifically about how Jupyter handles memory and files on your own computer, so do those two in Jupyter.

---
# 1. Which variables can a function see? (Scope)

**From Tyler's post:** You can create a function and use a variable inside it, but it is not always clear which variables a function can reach and which it cannot.

Every time a function runs, Python gives it a private scratch space. Names created inside the function, including its parameters, live in that scratch space and are thrown away when the function returns. Names created outside every function, at the top level of the notebook, are **global**.

When Python meets a name inside a function, it searches in this order and stops at the first match:

| Order | Scope | Where the name was created |
|---|---|---|
| 1 | **L**ocal | inside this function, including its parameters |
| 2 | **E**nclosing | inside a function that contains this one (rare for now) |
| 3 | **G**lobal | at the top level of the notebook or module |
| 4 | **B**uilt-in | names Python always has: `print`, `len`, `sum`, ... |

That search order is called the **LEGB rule**. Three rules cover almost everything you will run into:

1. **A function can read a global variable.**
2. **If a function assigns to a name anywhere in its body (`=`, `+=`), that name is local for the whole function.** A global with the same name is not touched.
3. **Code outside a function cannot see the function's local variables.** Values go in through parameters and come out through `return`.

### 1.1 Local variables disappear when the function returns

In [0]:
def make_greeting(name):
    message = f"Hello, {name}!"    # message is local to make_greeting
    return message

result = make_greeting("Scott")
print(result)

`result` exists because we stored what the function **returned**. `message` was the function's scratch work.

In [0]:
# SUPPOSED TO FAIL -- NameError
print(message)

The same is true for the parameter `name`: it only exists while `make_greeting` is running.

### 1.2 Reading a global works (Rule 1)

In [0]:
sales_tax = 0.08    # global: created outside any function

def price_with_tax(price):
    return round(price * (1 + sales_tax), 2)    # reads the global sales_tax

print(price_with_tax(50))

Python looks `sales_tax` up when the function **runs**, not when it is defined. **Predict** what happens if we change the global and call the function again:

In [0]:
sales_tax = 0.10
print(price_with_tax(50))

### 1.3 Assigning inside a function creates a new local (Rule 2)

**Predict** what the two `print` lines show before you run the cell.

In [0]:
counter = 0

def set_counter():
    counter = 99    # assignment -> a NEW local variable that happens to be named counter
    print("inside the function: ", counter)

set_counter()
print("outside the function:", counter)

Two different variables that share a name. The local one vanished when the function returned; the global one was never touched.

### 1.4 The confusing one: `UnboundLocalError`

This is the error that makes scope feel random. Read the function: it looks like it should add 1 to the global `counter`.

In [0]:
counter = 0

def add_one(counter): # JSP: Added counter as a parameter
    counter = counter + 1    # counter is assigned in this function, so it is local everywhere in it
    return counter

In [0]:
# SUPPOSED TO FAIL -- UnboundLocalError
'''
Note: I made a change as a test to see if we actually had to pass "counter" or if just getting a value assigned to the parameter counter was sufficient. This type of variable name overlap is terrible when you own the whole thing. Works well for libraries though.
'''
add_one(1) 


Python decides which names are local when it **reads the function definition**, before any line runs. It sees `counter = ...` in the body, so `counter` is local for the entire function, including the right-hand side `counter + 1`. When that line runs, the local `counter` does not have a value yet, and Python does not fall back to the global.

The fix is almost never "make it global." It is: **pass the value in, return the new value out.**

In [0]:
# Best fix: parameter in, return value out
def add_one(value):
    return value + 1

counter = 0
counter = add_one(counter)
counter = add_one(counter)
print(counter)

There is also a keyword that tells Python "inside this function, this name means the global one." You will see it in other people's code:

In [0]:
# Works, but use sparingly
counter = 0

def add_one_global():
    global counter          # counter now refers to the global counter
    counter = counter + 1

add_one_global() # = 1
add_one_global() # = 2
print(counter)

Why prefer the first version? With a parameter and `return`, everything the function depends on is visible in the call `add_one(counter)`. With `global`, the function secretly depends on (and changes) a variable defined somewhere else in the notebook. That is exactly the kind of code that breaks after a kernel restart or when you copy the function into another notebook.

### 1.5 The list surprise: changing an object is not assigning a name

Rule 2 is triggered by **assigning to the name**. Calling a method that changes a list or dictionary in place is not an assignment, so it changes the global object, no `global` keyword needed.

In [0]:
shopping = ["eggs"]

def add_item(item):
    shopping.append(item)    # no `shopping = ...` in this function, so shopping is the global list

add_item("milk")
print(shopping)

In [0]:
def replace_list():
    shopping = ["bread"]     # assignment -> brand-new local list; the global list is untouched
    shopping.append('butter') # JSP: Added to see if shopping was really a new list.
replace_list()
print(shopping)

| Inside a function, without `global` | Changes the global? | Why |
|---|---|---|
| `shopping.append("milk")` | Yes | changes the list object; no name is assigned |
| `shopping[0] = "tea"` | Yes | changes an item inside the list; the name `shopping` is not reassigned |
| `inventory["pens"] = 12` | Yes | same idea, for a dictionary |
| `shopping = ["bread"]` | No | assigns the name, so a new local variable is created |
| `shopping += ["bread"]` | Error | `+=` counts as assigning, so `shopping` is local and has no value yet (`UnboundLocalError`) |

### 1.6 Built-ins are the last place Python looks

`sum`, `len`, `max`, `list`, and `print` are built-in names. A variable with the same name is found **first** and hides the built-in.

In [0]:
def total_points(points):
    sum = 0                   # a local named sum hides the built-in sum() inside this function
    return sum(points)

In [0]:
# SUPPOSED TO FAIL -- TypeError
total_points([10, 20, 30])

`'int' object is not callable` means "you put parentheses after something that is a number, not a function."

In a notebook this usually happens at the top level: run `sum = 0` or `list = [1, 2, 3]` in any cell, and the built-in stays hidden **in every cell** until you restart the kernel (or run `del sum`). Pick names like `total` and `values` instead.

### 1.7 Why scope bugs hide in notebooks

Every variable you create at the top level of a notebook is global, and it stays in memory until the kernel restarts, even if you delete the cell that created it. So a function can quietly depend on a global you never meant to use.

In [0]:
def class_average():
    return sum(scores) / len(scores)    # scores is not a parameter...

scores = [88, 92, 79]
print(class_average())                  # ...but this works, because a global named scores exists

In [0]:
del scores

Now simulate a fresh kernel, or pasting that function into another notebook, where `scores` was never created:

In [0]:
# SUPPOSED TO FAIL -- NameError
del scores
print(class_average())

The fix is the same as in 1.4: make the input a parameter, so the function works no matter what else is (or is not) in memory.

In [0]:
def class_average(scores):
    return sum(scores) / len(scores)

print(class_average([88, 92, 79]))
print(class_average([70, 75]))

#### Your Turn 1.1 - Predict, then run

Write down what the three `print` lines will show. Then run the cell and compare.

In [0]:
level = "global"

def show_level():
    print("1:", level)

def change_level():
    level = "local"
    print("2:", level)

show_level()
change_level()
print("3:", level)

<details>
<summary><b>Show answer</b></summary>

```
1: global
2: local
3: global
```

- `show_level` only **reads** `level`, so it finds the global (Rule 1).
- `change_level` **assigns** `level`, so inside it `level` is a new local (Rule 2).
- The global was never changed, so line 3 still shows `global`.

</details>

#### Your Turn 1.2 - Fix the function

This function raises `UnboundLocalError`:

```python
total = 0

def add_to_total(amount):
    total = total + amount
```

Rewrite it **without** `global` so that this code prints `15`:

```python
total = 0
total = add_to_total(total, 10)
total = add_to_total(total, 5)
print(total)
```

In [0]:
# Your code here.
def add_to_total(total, amount):
    total += amount
    return total

total = 0
total = add_to_total(total, 10)
total = add_to_total(total, 5)
print(total)

<details>
<summary><b>Show answer</b></summary>

```python
def add_to_total(total, amount):
    return total + amount

total = 0
total = add_to_total(total, 10)
total = add_to_total(total, 5)
print(total)    # 15
```

The function no longer needs to know anything about the notebook. It receives the current total, returns the new one, and the caller decides where to store it.

</details>

#### Your Turn 1.3 - Which calls change the list?

Predict what `print(names)` shows, then run the cell. Use the table in 1.5 to explain each function.

In [0]:
names = ["Ana"]

def a():
    names.append("Ben")

def b():
    names = ["Chloe"]

def c():
    names[0] = "Dana"

a()
b()
c()
print(names)

<details>
<summary><b>Show answer</b></summary>

`['Dana', 'Ben']`

- `a()` changes the global list in place, so it becomes `['Ana', 'Ben']`.
- `b()` assigns the name `names`, which creates a local list inside `b`. The global list is untouched.
- `c()` replaces an **item** of the global list without assigning the name `names`, so it becomes `['Dana', 'Ben']`.

</details>

---
# 2. List comprehensions: from reading them to writing them

**From Tyler's post:** You can read a list comprehension, but writing one takes longer than writing a normal loop, and side-by-side comparisons with loops would help.

Good news first: **you never have to write a comprehension from scratch.** Write the loop you already know how to write, then translate it. The translation is mechanical, and after doing it twenty or so times you will start skipping the loop on your own.

A comprehension is a shortcut for exactly one loop shape: **start with an empty list, loop, maybe check a condition, append.**

```python
new_list = []
for ITEM in ITERABLE:
    if CONDITION:
        new_list.append(EXPRESSION)
```

becomes

```python
new_list = [EXPRESSION for ITEM in ITERABLE if CONDITION]
```

**The recipe:**

1. Write `[ ]`.
2. Inside it, first write what was inside `.append( )`.
3. Then copy the `for` line, without the colon.
4. Then copy the `if` line, without the colon (if there is one).

What you **keep** goes first. The `for` and `if` parts follow in the same top-to-bottom order they had in the loop.

### 2.1 Transform every item

In each cell below, the loop and the comprehension build the same list, and the last line checks that they match.

In [0]:
numbers = [1, 2, 3, 4, 5, 6]

# Loop
squares = []
for n in numbers:
    squares.append(n ** 2)

# Comprehension: what was inside .append( ), then the for line
squares_comp = [n ** 2 for n in numbers]

print(squares_comp)
print(squares == squares_comp)

### 2.2 Keep only some items (filter)

In [0]:
numbers = [1, 2, 3, 4, 5, 6]

# Loop
evens = []
for n in numbers:
    if n % 2 == 0:
        evens.append(n)

# Comprehension: the append part, then the for line, then the if line
evens_comp = [n for n in numbers if n % 2 == 0]

print(evens_comp)
print(evens == evens_comp)

When you keep each item unchanged, the front part is just the loop variable: `n for n in ...`. It looks repetitive, but the two `n`s have different jobs. The first one is "what to keep"; the second one names each item as the loop walks through the list.

### 2.3 Filter and transform together

In [0]:
names = ["al", "beatrice", "cy", "dominic", "eve"]

# Loop
long_names = []
for name in names:
    if len(name) > 3:
        long_names.append(name.title())

# Comprehension
long_names_comp = [name.title() for name in names if len(name) > 3]

print(long_names_comp)
print(long_names == long_names_comp)

### 2.4 `if`/`else`: the one that trips everyone up

A filter `if` (no `else`) decides **whether** an item is kept, so it goes at the **end**.

An `if`/`else` decides **what** to keep for every item, so it is part of the expression and goes at the **front**.

In [0]:
scores = [95, 62, 78, 49, 88]

# Loop: every score produces a label, so BOTH branches append
labels = []
for s in scores:
    if s >= 70:
        labels.append("pass")
    else:
        labels.append("fail")

# Comprehension: the if/else collapses into ONE expression, which takes the append position
labels_comp = ["pass" if s >= 70 else "fail" for s in scores]

print(labels_comp)
print(labels == labels_comp)

Read `"pass" if s >= 70 else "fail"` as a single value: *"pass" when the score is at least 70, otherwise "fail"*. That whole value is what gets appended, so it goes where the append part always goes.

Putting an `if`/`else` at the end is a syntax error:

In [0]:
# SUPPOSED TO FAIL -- SyntaxError
labels_comp = [s for s in scores if s >= 70 else "fail"]

| You want to... | Where the `if` goes | Example |
|---|---|---|
| keep **some** of the items | at the end, no `else` | `[s for s in scores if s >= 70]` |
| keep **every** item, choosing between two values | at the front, with `else` | `["pass" if s >= 70 else "fail" for s in scores]` |

### 2.5 Looping over pairs: `zip` and `enumerate`

Anything you can write after `for` in a loop, you can write after `for` in a comprehension, including two names at once.

In [0]:
unit_prices = [2.50, 10.00, 4.25]
quantities = [4, 1, 3]

# Loop
line_totals = []
for price, qty in zip(unit_prices, quantities):
    line_totals.append(price * qty)

# Comprehension
line_totals_comp = [price * qty for price, qty in zip(unit_prices, quantities)]

print(line_totals_comp)
print(line_totals == line_totals_comp)

In [0]:
runners = ["Kim", "Luis", "Maya"]

# Loop
podium = []
for place, runner in enumerate(runners, start=1):
    podium.append(f"{place}. {runner}")

# Comprehension
podium_comp = [f"{place}. {runner}" for place, runner in enumerate(runners, start=1)]

print(podium_comp)
print(podium == podium_comp)

### 2.6 Nested loops

With two `for` lines, copy them **in the same order** they appear in the loop: the outer loop first.

In [0]:
hours_by_week = [[3, 5, 2], [4, 6], [1, 7, 8]]

# Loop
all_hours = []
for week in hours_by_week:
    for h in week:
        all_hours.append(h)

# Comprehension: the append part, then the outer for, then the inner for
all_hours_comp = [h for week in hours_by_week for h in week]

print(all_hours_comp)
print(all_hours == all_hours_comp)

### 2.7 When to keep the loop

A comprehension only fits the "empty list, loop, maybe `if`, append" shape. Keep the regular loop when:

- each step depends on the previous one (running totals, "the best so far"),
- you need `break`, or several statements for each item,
- you are doing something for its effect, like `print`. `[print(x) for x in items]` runs, but it builds a useless list of `None` values.

In [0]:
deposits = [100, 250, 75]

# Each balance depends on the previous one, so a regular loop is the clear choice
balances = []
balance = 0
for amount in deposits:
    balance = balance + amount
    balances.append(balance)

print(balances)

One difference besides the syntax, and it connects back to Section 1: **a comprehension has its own scope.** The variable of a regular `for` loop still exists after the loop ends. The variable inside a comprehension does not exist outside the brackets.

In [0]:
for loop_item in [10, 20, 30]:
    pass

print(loop_item)    # the for-loop variable survives the loop

In [0]:
# SUPPOSED TO FAIL -- NameError
doubled = [comp_item * 2 for comp_item in [10, 20, 30]]
print(comp_item)

#### Your Turn 2.1 - Translate the loops

Each cell below has a working loop. Replace the empty list `[]` on the `_comp` line with a comprehension, and re-run the cell until the check prints `True`. Use the recipe: the append part first, then the `for`, then the `if`.

In [0]:
# a) The length of every word
words = ["data", "integrity", "audit", "py"]

lengths = []
for w in words:
    lengths.append(len(w))

lengths_comp = [len(w) for w in words]    # your comprehension here
print(lengths_comp == lengths)

In [0]:
# b) Only the prices under 20
item_prices = [12.99, 45.00, 5.50, 19.99, 20.00]

cheap = []
for p in item_prices:
    if p < 20:
        cheap.append(p)

cheap_comp = [p for p in item_prices if p < 20]    # your comprehension here
print(cheap_comp == cheap)

In [0]:
# c) 10% off anything over 50; everything else unchanged
item_prices = [80.0, 35.0, 120.0, 50.0]

sale = []
for p in item_prices:
    if p > 50:
        sale.append(p * 0.9)
    else:
        sale.append(p)

sale_comp = [p * 0.9 if p > 50 else p for p in item_prices]    # your comprehension here
print(sale_comp == sale)

In [0]:
# d) The capitalized first letter of every name that is not blank
signups = ["ana", "", "ben", "", "chloe"]

initials = []
for name in signups:
    if name != "":
        initials.append(name[0].upper())

initials_comp = [s[0].upper() for s in signups if s != '']    # your comprehension here
print(initials_comp == initials)

**e) No loop this time.** Build a list of every multiple of 3 from 1 to 30. If you get stuck, write the loop first, then translate it.

In [0]:
# Your code here.
threes = [(n * 3) for n in range(1, 11)]
print(threes)

**f) Now go backwards.** Rewrite this comprehension as a regular loop that builds the same list:

```python
emails = [" ANA@EXAMPLE.COM", "Ben@Example.com ", "not an email"]
clean = [e.strip().lower() for e in emails if "@" in e]
```

In [0]:
# Your code here.
emails = [" ANA@EXAMPLE.COM", "Ben@Example.com ", "not an email"]
clean = []

for e in emails:
    if '@' in e:
        clean.append(e.strip().lower())

print(clean)